# SHAP / XAI Analysis — Patrol Forecast v6
**Flipkart GridLock Hackathon 2.0** — 14 XAI Visualizations (300 DPI)

In [ ]:
import sys, os, warnings
import numpy as np
import pandas as pd
import joblib
import shap
import xgboost as xgb
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings('ignore')
%matplotlib inline
matplotlib.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Segoe UI', 'Arial', 'Helvetica'],
    'axes.titlesize': 16, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'savefig.facecolor': 'white', 'savefig.bbox': 'tight'
})

try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.path.abspath('')

MODEL_DIR = os.path.dirname(notebook_dir)
PROJECT_DIR = os.path.dirname(MODEL_DIR)
sys.path.insert(0, MODEL_DIR)
import features_v6 as F

PLOT_DIR = notebook_dir
os.makedirs(PLOT_DIR, exist_ok=True)
print(f'SHAP {shap.__version__} | XGBoost {xgb.__version__}')
print('Setup complete!')

In [ ]:
bundle = joblib.load(os.path.join(MODEL_DIR, 'production_v6.pkl'))
xgb_24h = bundle['24h']['xgb']
booster = xgb_24h.get_booster()

clean_df = F.load_clean(os.path.join(MODEL_DIR, 'seed_upload.csv'))
panel, keep, attrs = F.build_panel(clean_df)
panel['station'] = panel['station'].cat.set_categories(bundle['station_cats'])

latest = panel[panel.date == panel.date.max()].copy()
last_7d = panel[panel.date >= panel.date.max() - pd.Timedelta(days=7)].dropna(subset=['y_t1']).copy()

NUM_COLS = [c for c in F.FEATS_XGB if c != 'station']
def prepare_X(df):
    X = df[F.FEATS_XGB].copy()
    X[NUM_COLS] = X[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)
    return X

X_latest = prepare_X(latest)
sample_n = min(500, len(last_7d))
X_sample = prepare_X(last_7d.sample(n=sample_n, random_state=42))

X_sample_num = X_sample.copy()
for col in X_sample_num.select_dtypes(include='category').columns:
    X_sample_num[col] = X_sample_num[col].cat.codes.astype(float)

def compute_shap_native(X_df):
    dm = xgb.DMatrix(X_df, enable_categorical=True)
    contribs = booster.predict(dm, pred_contribs=True)
    base_val = float(contribs[0, -1])
    sv = contribs[:, :-1]
    return shap.Explanation(
        values=sv, base_values=np.full(len(sv), base_val),
        data=X_df.values, feature_names=list(X_df.columns)
    ), base_val

shap_latest, base_value = compute_shap_native(X_latest)
shap_sample, _ = compute_shap_native(X_sample)

mean_abs_shap = np.abs(shap_sample.values).mean(axis=0)
feat_ranking = np.argsort(mean_abs_shap)[::-1]
numeric_ranking = [i for i in feat_ranking if F.FEATS_XGB[i] in F.FEATS_NUM]
top1_feat = F.FEATS_XGB[numeric_ranking[0]]
top2_feat = F.FEATS_XGB[numeric_ranking[1]]
top3_feat = F.FEATS_XGB[numeric_ranking[2]]

pred_sums = shap_latest.values.sum(axis=1)
highest_idx = int(np.argmax(pred_sums))

print(f'X_latest: {X_latest.shape} | X_sample: {X_sample.shape}')
print(f'Base: {base_value:.4f} | Top: {top1_feat}, {top2_feat}, {top3_feat}')
print('Data & SHAP ready!')

---
## Graph 1: SHAP Beeswarm Summary

In [ ]:
plt.figure(figsize=(14, 10))
shap.plots.beeswarm(shap_sample, max_display=25, show=False)
plt.title('SHAP Beeswarm Plot \u2014 XGBoost 24h Violation Predictor', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('SHAP Value (impact on predicted violation count)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '1_shap_beeswarm_summary.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 1_shap_beeswarm_summary.png')

---
## Graph 2: Global Feature Importance Bar

In [ ]:
plt.figure(figsize=(12, 10))
shap.plots.bar(shap_sample, max_display=20, show=False)
plt.title('Global Feature Importance \u2014 Mean |SHAP| Values', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Mean |SHAP Value|', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '2_shap_feature_importance_bar.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 2_shap_feature_importance_bar.png')

---
## Graph 3: SHAP Dependence Scatter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 7))
plt.sca(axes[0])
shap.plots.scatter(shap_sample[:, top1_feat], show=False)
axes[0].set_title(f'Dependence: {top1_feat}', fontsize=14, fontweight='bold')
plt.sca(axes[1])
shap.plots.scatter(shap_sample[:, top2_feat], show=False)
axes[1].set_title(f'Dependence: {top2_feat}', fontsize=14, fontweight='bold')
fig.suptitle('SHAP Dependence Plots \u2014 Non-linear Feature Effects', fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '3_shap_dependence_scatter.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 3_shap_dependence_scatter.png')

---
## Graph 4: SHAP Waterfall

In [ ]:
plt.figure(figsize=(12, 10))
shap.plots.waterfall(shap_latest[highest_idx], max_display=15, show=False)
plt.title('Waterfall Plot \u2014 Highest-Risk Cell Prediction Breakdown', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '4_shap_waterfall_highrisk.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 4_shap_waterfall_highrisk.png')

---
## Graph 5: SHAP Force Plot

In [ ]:
sv = shap_latest.values[highest_idx]
fv = X_latest.iloc[highest_idx].values
fn = list(F.FEATS_XGB)
top_idx = np.argsort(np.abs(sv))[-15:]

plt.figure(figsize=(22, 4))
shap.force_plot(
    base_value=base_value, shap_values=sv[top_idx],
    features=fv[top_idx], feature_names=[fn[i] for i in top_idx],
    matplotlib=True, show=False, text_rotation=20
)
plt.title('Force Plot \u2014 Highest-Risk Prediction (Top 15 Features)', fontsize=14, fontweight='bold', pad=50)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '5_shap_force_plot.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 5_shap_force_plot.png')

---
## Graph 6: SHAP Heatmap

In [ ]:
n_hm = min(50, len(shap_latest))
sort_ord = np.argsort(shap_latest.values.sum(axis=1))[::-1][:n_hm]
plt.figure(figsize=(16, 10))
shap.plots.heatmap(shap_latest[sort_ord], max_display=20, show=False)
plt.title(f'SHAP Heatmap \u2014 Feature Contributions Across Top-{n_hm} Predictions', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '6_shap_heatmap.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 6_shap_heatmap.png')

---
## Graph 7: SHAP Decision Plot

In [ ]:
n_dec = min(30, len(shap_latest))
dec_order = np.argsort(pred_sums)[::-1][:n_dec]
plt.figure(figsize=(14, 12))
shap.decision_plot(
    base_value, shap_latest.values[dec_order],
    feature_names=list(F.FEATS_XGB),
    feature_display_range=slice(-20, None),
    highlight=0, show=False
)
plt.title('SHAP Decision Plot \u2014 Cumulative Feature Impact Paths', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '7_shap_decision_plot.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 7_shap_decision_plot.png')

---
## Graph 8: SHAP Violin Plot

In [ ]:
plt.figure(figsize=(14, 10))
shap.summary_plot(
    shap_sample.values, X_sample_num,
    feature_names=list(F.FEATS_XGB),
    plot_type='violin', max_display=20, show=False
)
plt.title('SHAP Violin Plot \u2014 Distribution of Feature Contributions', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('SHAP Value (impact on prediction)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '8_shap_violin_plot.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 8_shap_violin_plot.png')

---
## Graph 9: XAI Method Comparison (SHAP vs Gain vs Cover)

In [ ]:
gain_raw = booster.get_score(importance_type='gain')
cover_raw = booster.get_score(importance_type='cover')

bst_fnames = booster.feature_names
if bst_fnames is None:
    bst_fnames = [f'f{i}' for i in range(len(F.FEATS_XGB))]

def normalize_importance(raw_dict, feature_names):
    vals = np.array([raw_dict.get(fn, 0.0) for fn in feature_names])
    total = vals.sum()
    return vals / total if total > 0 else vals

shap_norm = mean_abs_shap / mean_abs_shap.sum()
gain_norm = normalize_importance(gain_raw, bst_fnames)
cover_norm = normalize_importance(cover_raw, bst_fnames)

top_n = 15
top_fi = feat_ranking[:top_n]
labels = [F.FEATS_XGB[i] for i in top_fi]

fig, ax = plt.subplots(figsize=(14, 8))
x = np.arange(top_n)
w = 0.25
ax.barh(x + w, shap_norm[top_fi], w, label='SHAP (Mean |SHAP|)', color='#FF006E', alpha=0.9)
ax.barh(x, gain_norm[top_fi], w, label='Gain (Split Improvement)', color='#3A86FF', alpha=0.9)
ax.barh(x - w, cover_norm[top_fi], w, label='Cover (Sample Reach)', color='#8338EC', alpha=0.9)

ax.set_yticks(x)
ax.set_yticklabels(labels, fontsize=11)
ax.invert_yaxis()
ax.set_xlabel('Normalized Importance', fontsize=12)
ax.set_title('XAI Method Comparison \u2014 SHAP vs Gain vs Cover', fontsize=16, fontweight='bold', pad=15)
ax.legend(fontsize=11, loc='lower right', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '9_xai_method_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 9_xai_method_comparison.png')

---
## Graph 10: Partial Dependence Plot (PDP)

In [ ]:
def compute_pdp(feature, X_bg, n_grid=50):
    vals = X_bg[feature].dropna()
    grid = np.linspace(vals.quantile(0.02), vals.quantile(0.98), n_grid)
    preds = []
    for g in grid:
        Xc = X_bg.copy()
        Xc[feature] = g
        dm = xgb.DMatrix(Xc, enable_categorical=True)
        preds.append(booster.predict(dm).mean())
    return grid, np.array(preds)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
top_feats = [top1_feat, top2_feat, top3_feat]
colors_pdp = ['#FF006E', '#3A86FF', '#00C49A']

for ax, feat, color in zip(axes, top_feats, colors_pdp):
    grid, preds = compute_pdp(feat, X_sample)
    ax.plot(grid, preds, color=color, linewidth=2.5, zorder=5)
    ax.fill_between(grid, preds.min(), preds, alpha=0.15, color=color)
    ax.set_xlabel(feat, fontsize=12, fontweight='bold')
    ax.set_ylabel('Predicted Violations', fontsize=11)
    ax.set_title(f'PDP: {feat}', fontsize=13, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, alpha=0.3)

fig.suptitle('Partial Dependence Plots \u2014 Marginal Feature Effects', fontsize=16, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '10_partial_dependence_plots.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 10_partial_dependence_plots.png')

---
## Graph 11: Cohort SHAP Comparison (High vs Low Risk)

In [ ]:
sums = shap_latest.values.sum(axis=1)
q_high = np.percentile(sums, 67)
q_low = np.percentile(sums, 33)
mask_high = sums >= q_high
mask_low = sums <= q_low

high_mean = np.abs(shap_latest.values[mask_high]).mean(axis=0)
low_mean = np.abs(shap_latest.values[mask_low]).mean(axis=0)

top_high_idx = np.argsort(high_mean)[::-1][:12]
top_low_idx = np.argsort(low_mean)[::-1][:12]
union_idx = list(dict.fromkeys(list(top_high_idx) + list(top_low_idx)))[:15]
labels_c = [F.FEATS_XGB[i] for i in union_idx]

fig, ax = plt.subplots(figsize=(14, 9))
y = np.arange(len(union_idx))
w = 0.35
ax.barh(y + w/2, high_mean[union_idx], w, label=f'High-Risk (top 33%, n={mask_high.sum()})',
        color='#FF006E', alpha=0.9, edgecolor='white', linewidth=0.5)
ax.barh(y - w/2, low_mean[union_idx], w, label=f'Low-Risk (bottom 33%, n={mask_low.sum()})',
        color='#3A86FF', alpha=0.9, edgecolor='white', linewidth=0.5)

ax.set_yticks(y)
ax.set_yticklabels(labels_c, fontsize=11)
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.set_title('Cohort Analysis \u2014 High-Risk vs Low-Risk Predictions', fontsize=16, fontweight='bold', pad=15)
ax.legend(fontsize=11, loc='lower right', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='x', alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '11_cohort_shap_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 11_cohort_shap_comparison.png')

---
## Graph 12: Feature Interaction Matrix

In [ ]:
n_interact = 20
top_interact_idx = feat_ranking[:n_interact]
top_interact_names = [F.FEATS_XGB[i] for i in top_interact_idx]

shap_subset = shap_sample.values[:, top_interact_idx]
shap_corr = np.corrcoef(shap_subset.T)

cmap = LinearSegmentedColormap.from_list('custom_div', ['#3A86FF', '#FFFFFF', '#FF006E'], N=256)

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.eye(n_interact, dtype=bool)
shap_corr_masked = np.where(mask, np.nan, shap_corr)
im = ax.imshow(shap_corr_masked, cmap=cmap, vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(n_interact))
ax.set_yticks(range(n_interact))
ax.set_xticklabels(top_interact_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(top_interact_names, fontsize=9)

for i in range(n_interact):
    for j in range(n_interact):
        if i != j:
            val = shap_corr[i, j]
            color = 'white' if abs(val) > 0.5 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=6.5, color=color, fontweight='bold' if abs(val) > 0.4 else 'normal')

cbar = plt.colorbar(im, ax=ax, shrink=0.8, label='SHAP Value Correlation')
ax.set_title('Feature Interaction Matrix \u2014 SHAP Value Correlations (Top 20)', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '12_feature_interaction_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 12_feature_interaction_matrix.png')

---
## Graph 13: SHAP Directional Impact (Positive vs Negative)
Shows the **average direction** each feature pushes predictions.  
Red bars = features that increase violations on average. Blue bars = features that suppress violations.

In [ ]:
# Mean SHAP (signed) for top 20 features
n_dir = 20
dir_idx = feat_ranking[:n_dir]
mean_signed = shap_sample.values[:, dir_idx].mean(axis=0)
mean_abs = mean_abs_shap[dir_idx]
dir_names = [F.FEATS_XGB[i] for i in dir_idx]

# Split into positive and negative contributions
pos_mean = np.where(shap_sample.values[:, dir_idx] > 0, shap_sample.values[:, dir_idx], 0).mean(axis=0)
neg_mean = np.where(shap_sample.values[:, dir_idx] < 0, shap_sample.values[:, dir_idx], 0).mean(axis=0)

fig, ax = plt.subplots(figsize=(14, 10))
y = np.arange(n_dir)

# Positive (right, red) and negative (left, blue) bars
ax.barh(y, pos_mean, height=0.7, color='#FF006E', alpha=0.85, label='Increases violations (+)', edgecolor='white', linewidth=0.5)
ax.barh(y, neg_mean, height=0.7, color='#3A86FF', alpha=0.85, label='Decreases violations (\u2212)', edgecolor='white', linewidth=0.5)

# Add zero line
ax.axvline(0, color='#333333', linewidth=1.2, zorder=3)

# Net direction arrows
for i, (p, n, s) in enumerate(zip(pos_mean, neg_mean, mean_signed)):
    net_x = max(abs(p), abs(n)) + 0.005
    arrow = '\u25B6' if s > 0 else '\u25C0' if s < 0 else '\u25CF'
    clr = '#FF006E' if s > 0 else '#3A86FF'
    ax.text(net_x, i, f' {arrow} net: {s:+.3f}', va='center', fontsize=8.5, color=clr, fontweight='bold')

ax.set_yticks(y)
ax.set_yticklabels(dir_names, fontsize=11)
ax.invert_yaxis()
ax.set_xlabel('Mean SHAP Value', fontsize=12)
ax.set_title('SHAP Directional Impact \u2014 Which Features Push Violations UP vs DOWN?',
             fontsize=15, fontweight='bold', pad=15)
ax.legend(fontsize=11, loc='lower right', framealpha=0.9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, axis='x', alpha=0.15)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '13_shap_directional_impact.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 13_shap_directional_impact.png')

---
## Graph 14: Cumulative Feature Importance (Pareto Analysis)
Shows how many features are needed to explain a given % of total model importance.  
Reveals the **information concentration** \u2014 typically a few features drive most of the prediction.

In [ ]:
# Sort features by importance and compute cumulative %
sorted_imp = mean_abs_shap[feat_ranking]
total_imp = sorted_imp.sum()
cumulative_pct = np.cumsum(sorted_imp) / total_imp * 100
n_feats = len(sorted_imp)

fig, ax1 = plt.subplots(figsize=(14, 8))

# Bar chart of individual importance
x = np.arange(min(30, n_feats))
bar_vals = sorted_imp[:30] / total_imp * 100
bar_names = [F.FEATS_XGB[feat_ranking[i]] for i in range(min(30, n_feats))]

colors_bar = ['#FF006E' if cumulative_pct[i] <= 80 else '#FFBE0B' if cumulative_pct[i] <= 95 else '#AAAAAA'
              for i in range(len(x))]
ax1.bar(x, bar_vals, color=colors_bar, alpha=0.85, edgecolor='white', linewidth=0.5)
ax1.set_ylabel('Individual Importance (%)', fontsize=12, color='#FF006E')
ax1.tick_params(axis='y', labelcolor='#FF006E')
ax1.set_xticks(x)
ax1.set_xticklabels(bar_names, rotation=55, ha='right', fontsize=8.5)

# Cumulative line on secondary axis
ax2 = ax1.twinx()
ax2.plot(x, cumulative_pct[:30], color='#3A86FF', linewidth=2.5, marker='o', markersize=5, zorder=5)
ax2.set_ylabel('Cumulative Importance (%)', fontsize=12, color='#3A86FF')
ax2.tick_params(axis='y', labelcolor='#3A86FF')
ax2.set_ylim(0, 105)

# Reference lines at 80% and 95%
for pct, ls, lbl in [(80, '--', '80%'), (95, ':', '95%')]:
    ax2.axhline(pct, color='#888888', linestyle=ls, linewidth=1, alpha=0.7)
    # Find how many features reach this threshold
    n_at_pct = int(np.searchsorted(cumulative_pct, pct)) + 1
    ax2.text(min(29, n_at_pct + 0.5), pct + 1.5, f'{lbl} at {n_at_pct} features',
             fontsize=10, color='#555555', fontweight='bold')

ax1.set_title('Cumulative Feature Importance \u2014 Pareto Analysis\n'
              f'({n_feats} total features, top features drive most predictions)',
              fontsize=15, fontweight='bold', pad=15)

# Legend
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Patch(facecolor='#FF006E', alpha=0.85, label='Top 80% (core drivers)'),
    Patch(facecolor='#FFBE0B', alpha=0.85, label='80-95% (supporting)'),
    Patch(facecolor='#AAAAAA', alpha=0.85, label='>95% (marginal)'),
    Line2D([0], [0], color='#3A86FF', linewidth=2.5, marker='o', label='Cumulative %')
]
ax1.legend(handles=legend_elements, fontsize=10, loc='center right', framealpha=0.9)

ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '14_cumulative_importance_pareto.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 14_cumulative_importance_pareto.png')

---
## Summary

In [ ]:
print('=' * 60)
print('  ALL 14 XAI GRAPHS GENERATED')
print('=' * 60)
for f in sorted(os.listdir(PLOT_DIR)):
    if f.endswith('.png'):
        sz = os.path.getsize(os.path.join(PLOT_DIR, f)) / 1024
        print(f'  {f:50s} ({sz:6.0f} KB)')
print(f'\nModel: XGBoost Poisson 24h | Method: Native TreeSHAP')
print(f'Features: {len(F.FEATS_XGB)} | Base: {base_value:.3f}')